# Cumulative Mismatch Penalty

The oracle uses **MISMATCH_DECAY = 0.87**: each additional imperfect feature multiplies the score. So `final_score = base_score × 0.87^num_mismatches`.

- **1 mismatch:** 0.87× (13% drop)
- **4 mismatches:** 0.87⁴ ≈ 0.57 (nearly halves the score)
- **6 mismatches:** 0.87⁶ ≈ 0.44

**Hypothesis:** Multiple small mismatches together are worse than the sum of their individual effects. The model should learn this compound penalty. We verify by grouping interactions by `num_mismatches` and comparing survival rate (and model score) vs the expected decay curve.

## 1. Setup & Load Data

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "resources").exists():
    for c in [ROOT.parent, ROOT.parent.parent, ROOT.parent.parent.parent]:
        if (c / "resources").exists():
            ROOT = c
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERACTIONS_PATH = ROOT / "resources" / "data" / "synthetic_interactions.json"
USERS_PATH = ROOT / "resources" / "data" / "synthetic_users.json"
PLANTS_PATH = ROOT / "resources" / "data_creating" / "permapeople_plants_mapped_normalized.json"
PLANTS_ALT = ROOT / "resources" / "data_creating" / "permapeople_plants_mapped_uniform.json"
OUTPUT_DIR = ROOT / "resources" / "verify" / "cumulative mismatch"
MISMATCH_DECAY = 0.87  # from generate_interactions.py

In [ ]:
LIGHT_ORDER = ["full shade", "partial sun/shade", "full sun"]
SOIL_ORDER = ["light", "medium", "heavy"]
CARE_ORDER = ["easy", "medium", "hard"]
CLIMATE_ORDER = ["alpine", "arid", "mediterranean", "temperate", "tropical"]

def _norm(v):
    if v is None: return None
    return str(v).strip().lower() or None

def _plant_ec(plant: dict, key: str, alt: str | None = None) -> dict:
    ec = plant.get("environment_care") or {}
    return ec.get(key) or ec.get(alt or key) or {}

def _plant_val(plant: dict, key: str) -> str | None:
    ec = plant.get("environment_care") or {}
    return ec.get(key)

def _extract_plant_usda_zone(plant: dict) -> tuple[int, int]:
    ec = plant.get("environment_care") or {}
    min_z, max_z = ec.get("usda_zone_min"), ec.get("usda_zone_max")
    if min_z is not None and max_z is not None:
        return max(1, min(13, int(min_z))), max(1, min(13, int(max_z)))
    usda = ec.get("USDA Hardiness zone")
    if isinstance(usda, dict):
        min_z = usda.get("min", 3)
        max_z = usda.get("max", 9)
        return max(1, min(13, int(min_z))), max(1, min(13, int(max_z)))
    return (3, 9)

def _ordinal_distance(user_val, plant_vals: list, order: list) -> float:
    if not user_val or not plant_vals: return 0.0
    uv = _norm(user_val)
    if not uv: return 0.0
    rank_map = {_norm(v): i for i, v in enumerate(order)}
    user_rank = rank_map.get(uv)
    if user_rank is None: return 0.0
    plant_ranks = [rank_map.get(_norm(p)) for p in plant_vals if p]
    plant_ranks = [r for r in plant_ranks if r is not None]
    if not plant_ranks: return 0.0
    if user_rank in plant_ranks: return 0.0
    return min(abs(user_rank - r) for r in plant_ranks)

def _zone_overlap(user_min, user_max, plant_min, plant_max) -> float:
    overlap = max(0, min(user_max, plant_max) - max(user_min, plant_min) + 1)
    span = user_max - user_min + 1
    return overlap / span if span > 0 else 0.0

def feature_match(interaction: dict, user: dict, plant: dict) -> dict[str, bool]:
    """Return dict of feature_name -> True if match, False if mismatch."""
    ec = plant.get("environment_care") or {}
    result = {}
    lr = _plant_ec(plant, "lighting", "light_req")
    ideal, toler = lr.get("ideal_light"), lr.get("tolerated_light") or lr.get("ideal_light")
    plant_lights = [v for v in (ideal, toler) if v]
    result["light"] = _ordinal_distance(user.get("light"), plant_lights, LIGHT_ORDER) == 0.0
    from resources.ETL.feature_engineer import water_freq_to_days
    wr = _plant_ec(plant, "water", "water_req")
    ideal_d = wr.get("ideal_water_days") or water_freq_to_days(wr.get("ideal_water"))
    toler_d = wr.get("tolerated_water_days") or water_freq_to_days(wr.get("tolerated_water"))
    p_min = min(ideal_d, toler_d) if ideal_d and toler_d else 2
    p_max = max(ideal_d, toler_d) if ideal_d and toler_d else 2
    uf = user.get("water_freq")
    result["water"] = (uf is not None and p_min <= float(uf) <= p_max) if uf is not None else True
    sr = _plant_ec(plant, "soil", "soil_req")
    ideal_s, toler_s = sr.get("ideal_soil"), sr.get("tolerated_soil") or sr.get("ideal_soil")
    plant_soils = [v for v in (ideal_s, toler_s) if v]
    result["soil"] = _ordinal_distance(user.get("soil"), plant_soils, SOIL_ORDER) == 0.0
    pc = _norm(plant.get("care_level") or ec.get("care_level")) or "medium"
    uc = _norm(user.get("care_level"))
    rank = {_norm(v): i for i, v in enumerate(CARE_ORDER)}
    ur, pr = rank.get(uc) if uc else None, rank.get(pc)
    result["care"] = (ur is not None and pr is not None and ur >= pr) if (ur is not None and pr is not None) else True
    plc = _norm(_plant_val(plant, "origin_climate"))
    result["climate"] = _ordinal_distance(user.get("climate"), [plc] if plc else [], CLIMATE_ORDER) == 0.0
    u_min, u_max = user.get("usda_zone_min"), user.get("usda_zone_max")
    if u_min is not None and u_max is not None:
        p_min, p_max = _extract_plant_usda_zone(plant)
        result["zone"] = _zone_overlap(int(u_min), int(u_max), p_min, p_max) > 0
    else:
        result["zone"] = True
    return result

In [ ]:
def load_data():
    with open(INTERACTIONS_PATH) as f:
        interactions = json.load(f)
    with open(USERS_PATH) as f:
        users = json.load(f)
    path = PLANTS_PATH if PLANTS_PATH.exists() else PLANTS_ALT
    with open(path) as f:
        plants = json.load(f).get("plants", [])
    user_by_id = {u["user_id"]: u for u in users}
    plant_by_id = {p.get("id"): p for p in plants if p.get("id") is not None}
    return interactions, user_by_id, plant_by_id

interactions, user_by_id, plant_by_id = load_data()
print(f"{len(interactions)} interactions, {len(user_by_id)} users, {len(plant_by_id)} plants")

## 2. Survival by Number of Mismatches

Group interactions by `num_mismatches` (0–6) and compute survival rate. Oracle: `score × 0.87^num_mismatches` — so survival should drop as mismatches compound.

In [ ]:
def compute_survival_by_num_mismatches(interactions, user_by_id, plant_by_id):
    """Group by num_mismatches; return {n: {"n": count, "survival_rate": ...}}."""
    by_n = defaultdict(list)
    for r in interactions:
        u = user_by_id.get(r["user_id"], {})
        p = plant_by_id.get(r["plant_id"]) or {}
        if not p:
            continue
        matches = feature_match(r, u, p)
        num_mismatches = sum(1 for m in matches.values() if not m)
        by_n[num_mismatches].append(r.get("label", 0))
    return {
        n: {
            "n": len(labels),
            "survival_rate": sum(labels) / len(labels) if labels else 0,
        }
        for n, labels in sorted(by_n.items())
    }

stats_by_n = compute_survival_by_num_mismatches(interactions, user_by_id, plant_by_id)
for n, s in stats_by_n.items():
    print(f"{n} mismatches: survival={s['survival_rate']:.3f} (n={s['n']})")

## 3. Expected Decay Curve vs Observed

Oracle: `final_score = base_score × 0.87^num_mismatches`. The multiplier alone (ignoring base_score variance) shows how each extra mismatch compounds. 4 mismatches → 0.87⁴ ≈ 0.57 (nearly halves).

In [ ]:
import matplotlib.pyplot as plt

n_vals = sorted(stats_by_n.keys())
survival_rates = [stats_by_n[n]["survival_rate"] for n in n_vals]
counts = [stats_by_n[n]["n"] for n in n_vals]
decay_curve = [MISMATCH_DECAY ** n for n in n_vals]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="white")

# Left: survival rate + decay curve (normalize decay to same scale as survival for comparison)
axes[0].bar(n_vals, survival_rates, color="#2e7d32", alpha=0.7, label="Survival rate")
# Scale decay to overlap [0,1] with survival for visual comparison
decay_scaled = [d * (survival_rates[0] / decay_curve[0]) if decay_curve[0] > 0 else d for d in decay_curve]
axes[0].plot(n_vals, decay_scaled, "o-", color="#1565c0", linewidth=2, label="0.87^n (scaled to match)")
axes[0].set_xlabel("Number of mismatches")
axes[0].set_ylabel("Survival rate / Scaled decay")
axes[0].set_title("Cumulative mismatch: survival vs 0.87^n decay")
axes[0].legend()
axes[0].set_xticks(n_vals)

# Right: raw decay multiplier (what oracle applies)
axes[1].bar(n_vals, decay_curve, color="#7e57c2", alpha=0.7)
axes[1].axhline(0.5, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Number of mismatches")
axes[1].set_ylabel("Decay multiplier (0.87^n)")
axes[1].set_title("Oracle penalty: each mismatch × 0.87")
for i, (n, d) in enumerate(zip(n_vals, decay_curve)):
    axes[1].annotate(f"{d:.2f}", (n, d), ha="center", va="bottom", fontsize=9)
axes[1].set_xticks(n_vals)

plt.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_DIR / "cumulative_mismatch_survival.png", dpi=120, bbox_inches="tight", facecolor="white")
plt.show()

## 4. Model: Does It Learn the Compound Penalty?

Run model inference and group scores by `num_mismatches`. If the model learned the pattern, mean score should decrease as mismatches increase (compound effect), not just linearly.

In [ ]:
import torch
from resources.ETL.feature_engineer import apply_user_embeddings, apply_categorical_embeddings

users_list = list(user_by_id.values())
plants_list = list(plant_by_id.values())
apply_user_embeddings(users_list)
apply_categorical_embeddings(plants_list)
user_by_id = {u["user_id"]: u for u in users_list}
plant_by_id = {p.get("id"): p for p in plants_list}

model = None
for p in [ROOT / "resources" / "two_tower_training" / "two_tower.pt",
          ROOT / "resources" / "two_tower_training" / "output" / "two_tower.pt"]:
    if p.exists():
        ckpt = torch.load(p, map_location="cpu", weights_only=True)
        state = ckpt.get("model_state", ckpt)
        if any("user_tower.mlp" in k for k in state.keys()):
            from resources.two_tower_training.two_tower_model import TwoTowerModel
            model = TwoTowerModel()
            model.load_state_dict(state, strict=True)
            model.eval()
            print(f"Loaded model from {p}")
            break

In [ ]:
def compute_model_scores_by_num_mismatches(interactions, user_by_id, plant_by_id, model):
    """Bucket by num_mismatches; return mean model score per bucket."""
    batch_u, batch_p, batch_n = [], [], []
    for r in interactions:
        u = user_by_id.get(r["user_id"], {})
        p = plant_by_id.get(r["plant_id"]) or {}
        if not p or "categorical_embedding" not in p:
            continue
        u_emb = u.get("categorical_embedding")
        if not u_emb or len(u_emb) != 21:
            continue
        matches = feature_match(r, u, p)
        num_mismatches = sum(1 for m in matches.values() if not m)
        batch_u.append(list(u_emb) + [0.0] * 64 + [0.0] * 64)
        batch_p.append(p["categorical_embedding"])
        batch_n.append(num_mismatches)
    if not batch_u:
        return None
    u_t = torch.tensor(batch_u, dtype=torch.float32)
    p_t = torch.tensor(batch_p, dtype=torch.float32)
    with torch.no_grad():
        scores = model(u_t, p_t).numpy()
    by_n = defaultdict(list)
    for n, sc in zip(batch_n, scores):
        by_n[n].append(float(sc))
    return {n: {"n": len(v), "mean_score": sum(v) / len(v)} for n, v in sorted(by_n.items())}

model_stats_by_n = compute_model_scores_by_num_mismatches(interactions, user_by_id, plant_by_id, model) if model else None
if model_stats_by_n:
    for n, s in model_stats_by_n.items():
        print(f"{n} mismatches: mean_score={s['mean_score']:.4f} (n={s['n']})")
else:
    print("Model not loaded — skip model analysis.")

In [ ]:
if model_stats_by_n:
    fig2, ax2 = plt.subplots(figsize=(10, 5), facecolor="white")
    n_vals = sorted(model_stats_by_n.keys())
    model_scores = [model_stats_by_n[n]["mean_score"] for n in n_vals]
    survival = [stats_by_n.get(n, {}).get("survival_rate", 0) for n in n_vals]
    ax2_twin = ax2.twinx()
    ax2.bar([x - 0.2 for x in n_vals], survival, 0.4, label="Survival rate", color="#2e7d32", alpha=0.8)
    ax2_twin.bar([x + 0.2 for x in n_vals], model_scores, 0.4, label="Mean model score", color="#1565c0", alpha=0.8)
    ax2.set_xlabel("Number of mismatches")
    ax2.set_ylabel("Survival rate", color="#2e7d32")
    ax2_twin.set_ylabel("Mean model score", color="#1565c0")
    ax2.tick_params(axis="y", labelcolor="#2e7d32")
    ax2_twin.tick_params(axis="y", labelcolor="#1565c0")
    ax2.set_title("Model learns compound penalty: score drops as mismatches increase")
    ax2.legend(loc="upper left")
    ax2_twin.legend(loc="upper right")
    ax2.set_xticks(n_vals)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "cumulative_mismatch_model.png", dpi=120, bbox_inches="tight", facecolor="white")
    plt.show()

## 5. Summary

- **Oracle:** `0.87^num_mismatches` — 4 mismatches ≈ 0.57× (nearly halves). Multiple small mismatches compound.
- **Survival:** Observed survival rate should decrease as `num_mismatches` increases (data confirms oracle).
- **Model:** If the model learned the compound penalty, mean score should also drop with more mismatches. Compare the curve shape: steeper drop (compound) vs linear drop (additive).